Prima parte:

1. Si consideri il dataset clinico SUPPORT2, contenente informazioni demografiche, fisiologiche e di laboratorio 
relative a pazienti ospedalizzati. Dopo aver caricato i dati, rimuovere le variabili di outcome e prognosi per 
evitare fenomeni di data leakage e selezionare una variabile target (ad esempio dzgroup).
2. Suddividere il dataset in training set (95%) e test set (5%) utilizzando un campionamento stratificato rispetto 
alla variabile target («death»), in modo da preservare la distribuzione delle classi.
3. Gestire i dati mancanti applicando imputazione con mediana per le variabili numeriche e con un valore 
costante (ad esempio "Unknown") per quelle categoriche; successivamente codificare le variabili categoriche 
tramite Ordinal Encoding.
4. Applicare lo scaling delle feature numeriche tramite z-scoring, quindi calcolare e analizzare la matrice di 
correlazione, individuando eventuali coppie di variabili altamente correlate e discutendo la presenza di 
multicollinearità.
5. Gestire la multicollinearità mediante rimozione di feature ridondanti o tramite una trasformazione lineare (ad 
esempio PCA), e infine applicare un metodo di selezione delle feature per identificare le variabili più rilevanti, 
commentando criticamente i risultati ottenuti.
Esercizi da svolgere


seconda parte:

5. Implementare un clustering dei punti del data set processato nelle fasi precedenti, utilizzando l’algoritmo 
DBSCAN con una griglia di ricerca degli iper-parametri:
MinPts = 5, 10, 20
Eps = 0.5, 1.5, 2.0
Stampare l’accuracy del modello di clustering sul training set e sul test set. Valuta la purezza del clustering.
6. Implementare un clustering dei punti del dataset processato nelle fasi precedenti, utilizzando un secondo 
algoritmo di clustering. La selezione dell’algoritmo deve essere coerente con la natura del dato. Valuta la 
purezza del clustering. 
7. Esegui la comparazione delle performance utilizzando l’etichetta «death» con criteri di valutazione esterni.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
df = pd.read_csv('./dataset_esercitazione.csv')

deathCol = df['death'] 

df.drop( columns=['dzgroup', 'dzclass','death'], inplace=True)
X_tr, X_te = train_test_split(df, test_size=0.2, random_state=42, stratify=deathCol) 

y_tr = deathCol.loc[X_tr.index]
y_te = deathCol.loc[X_te.index]

In [24]:
numeric_features = X_tr.select_dtypes(include=['number']).columns.tolist()

cat_features = [ column  for column in X_tr.columns if column not in numeric_features]


import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

# 1. IMPUTATION
imputer1 = SimpleImputer(strategy='median').set_output(transform="pandas")
imputer2 = SimpleImputer(strategy='constant', fill_value='Unknown').set_output(transform="pandas")

# Fit e transform separati per tipo
X_tr_num = imputer1.fit_transform(X_tr[numeric_features])
X_tr_cat = imputer2.fit_transform(X_tr[cat_features])

X_te_num = imputer1.transform(X_te[numeric_features])
X_te_cat = imputer2.transform(X_te[cat_features])

# 2. ENCODING (Solo sulle colonne categoriche!)
encoder = OrdinalEncoder().set_output(transform="pandas")

# Applichiamo l'encoder SOLO ai pezzi categorici già imputati
X_tr_cat_enc = encoder.fit_transform(X_tr_cat)
X_te_cat_enc = encoder.transform(X_te_cat)

# 3. UNIONE FINALE (Numeri originali + Categorie encodate)
X_train_final = pd.concat([X_tr_num, X_tr_cat_enc], axis=1)
X_test_final = pd.concat([X_te_num, X_te_cat_enc], axis=1)

print(X_train_final.head())

           age  num.co   edu  scoma   charges       totcst      totmcst  \
8154  75.09399     2.0   8.0    9.0   15305.0   10673.5938   8756.25000   
3129  75.27399     3.0  12.0   26.0   25024.0   14452.7344  13202.25000   
6170  19.35100     1.0  12.0    0.0  139899.0   98136.8125  96743.87500   
605   53.24597     1.0  12.0   26.0  231587.0  112461.6880  13202.25000   
6903  61.50299     1.0  10.0    0.0   11624.0    7809.5000   8441.59375   

      avtisst        sps   aps  ...   bun   urine  adlp  adls  adlsc  sex  \
8154     33.0  24.699219  41.0  ...  23.0  1968.0   0.0   0.0    0.0  1.0   
3129     48.0  29.699219  68.0  ...  41.0   685.0   0.0   0.0    0.0  1.0   
6170     18.5  31.597656  38.0  ...  23.0  1968.0   0.0   0.0    0.0  1.0   
605      45.0  39.695312  76.0  ...  23.0  1968.0   0.0   5.0    5.0  1.0   
6903     22.5  20.500000  28.0  ...  13.0  1950.0   0.0   1.0    1.0  0.0   

      income  race   ca  dnr  
8154     4.0   5.0  2.0  1.0  
3129     1.0   5.0  1.0 

In [25]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_final[numeric_features] = scaler.fit_transform(X_train_final[numeric_features])
X_test_final[numeric_features] = scaler.transform(X_test_final[numeric_features])
print(X_train_final.head())


           age    num.co       edu     scoma   charges    totcst   totmcst  \
8154  0.793589  0.097031 -1.222335 -0.128930 -0.434594 -0.425634 -0.396803   
3129  0.805125  0.837884  0.057591  0.553518 -0.340237 -0.340078 -0.274256   
6170 -2.778896 -0.643822  0.057591 -0.490227  0.775026  1.554439  2.028448   
605  -0.606618 -0.643822  0.057591  0.553518  1.665178  1.878739 -0.274256   
6903 -0.077438 -0.643822 -0.582372 -0.490227 -0.470331 -0.490474 -0.405476   

       avtisst       sps       aps  ...       bun     urine      adlp  \
8154  0.792216 -0.087061  0.170702  ... -0.241808 -0.111153 -0.363265   
3129  1.933891  0.416104  1.520421  ...  0.663243 -1.385282 -0.363265   
6170 -0.311402  0.607150  0.020733  ... -0.241808 -0.111153 -0.363265   
605   1.705556  1.422042  1.920338  ... -0.241808 -0.111153 -0.363265   
6903 -0.006956 -0.509642 -0.479163  ... -0.744615 -0.129029 -0.363265   

          adls     adlsc  sex  income  race   ca  dnr  
8154 -0.559645 -0.936817  1.0     4.

In [29]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(f_classif,k=18)
x_train_selected = selector.fit_transform(X_train_final, y_tr)
x_test_selected = selector.transform(X_test_final)

print(x_train_selected.shape)


(7284, 18)


5. Implementare un clustering dei punti del data set processato nelle fasi precedenti, utilizzando l’algoritmo 
DBSCAN con una griglia di ricerca degli iper-parametri:
MinPts = 5, 10, 20
Eps = 0.5, 1.5, 2.0
Stampare l’accuracy del modello di clustering sul training set e sul test set. Valuta la purezza del clustering.

In [ ]:
#Clustering con DBScan


from sklearn.cluster import DBSCAN

clusters = []

for eps in [0.5, 1.0, 1.5]:
    for min_samples in [5, 10, 15]:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        cluster_labels = dbscan.fit_predict(x_train_selected)
        clusters.append((eps, min_samples, cluster_labels))
    


[-1 -1 -1 ... -1 -1 -1]
[-1 -1 -1 ... -1 -1 -1]
[-1 -1 -1 ... -1 -1 -1]
[-1 -1 -1 ... -1 -1 -1]
[-1 -1 -1 ... -1 -1 -1]
[-1 -1 -1 ... -1 -1 -1]
[-1 -1 -1 ...  0  0 -1]
[-1 -1 -1 ... -1  0 -1]
[-1 -1 -1 ... -1  0 -1]


In [ ]:

#verifica quanti punti sono stati classificati come rumore (-1) e quanti cluster sono stati identificati (escluso il rumore)

for eps, min_samples, cluster_labels in clusters:
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    n_non_noise = (cluster_labels != -1).sum()
    print(f"eps={eps}, min_samples={min_samples} -> cluster: {n_clusters}, punti in cluster: {n_non_noise}")

eps=0.5, min_samples=5 -> cluster: 0, punti in cluster: 0
eps=0.5, min_samples=10 -> cluster: 0, punti in cluster: 0
eps=0.5, min_samples=15 -> cluster: 0, punti in cluster: 0
eps=1.0, min_samples=5 -> cluster: 6, punti in cluster: 768
eps=1.0, min_samples=10 -> cluster: 8, punti in cluster: 381
eps=1.0, min_samples=15 -> cluster: 4, punti in cluster: 130
eps=1.5, min_samples=5 -> cluster: 14, punti in cluster: 2991
eps=1.5, min_samples=10 -> cluster: 4, punti in cluster: 2580
eps=1.5, min_samples=15 -> cluster: 1, punti in cluster: 2332


In [35]:


accuracies = []

for eps, min_samples, cluster_labels in clusters:
    temp = pd.DataFrame({"cluster": cluster_labels, "y": y_tr.values})

    # assegna a ogni cluster la classe maggioritaria
    cluster_to_class = {
        cl: temp[temp["cluster"] == cl]["y"].mode().iloc[0]
        for cl in temp["cluster"].unique()
        if cl != -1
    }

    # traduce i cluster in etichette di classe
    y_pred = temp["cluster"].map(cluster_to_class)

    # i punti rumore vengono assegnati alla classe più frequente del dataset
    y_pred = y_pred.fillna(y_tr.mode().iloc[0])

    acc = (y_pred.values == y_tr.values).mean()

    accuracies.append((eps, min_samples, acc))

for eps, min_samples, acc in accuracies:
    print(f"eps={eps}, min_samples={min_samples} -> accuracy={acc:.4f}")

eps=0.5, min_samples=5 -> accuracy=0.6811
eps=0.5, min_samples=10 -> accuracy=0.6811
eps=0.5, min_samples=15 -> accuracy=0.6811
eps=1.0, min_samples=5 -> accuracy=0.6875
eps=1.0, min_samples=10 -> accuracy=0.6829
eps=1.0, min_samples=15 -> accuracy=0.6811
eps=1.5, min_samples=5 -> accuracy=0.6833
eps=1.5, min_samples=10 -> accuracy=0.6811
eps=1.5, min_samples=15 -> accuracy=0.6811
